In [1]:
from models.GeoTKG import GeoTKG
import torch
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from models.TIEUtils import collator as tie_collator, TemporalDataset
from models.GeoUtils import collator as geo_collator, GeoDataset
from models.TimexNormUtils import collator as norm_collator
from torch.utils.data import DataLoader
from torch.utils.data import Sampler

class EpochRandomSubsetSampler(Sampler):
    def __init__(self, data_source, subset_size):
        self.data_source = data_source
        self.subset_size = subset_size

    def __iter__(self):
        # Pick a new random subset each epoch
        indices = torch.randperm(len(self.data_source))[:self.subset_size]
        return iter(indices.tolist())

    def __len__(self):
        return self.subset_size


d:\GeoTKG\venv\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
NUM_EPOCHS = 20
ENC_LR = 5e-5
NONENC_LR = 1e-3
WARMUP_EPOCHS = 5
LOGGING_EPOCHS = 5
WEIGHT_DECAY = 0.01
BATCH_SIZE = 4
EEPOCH_SIZE = 5400

In [3]:
cleandata_path = "D:\\GeoTKG\\cleandata\\tie\\"
tie_train = TemporalDataset(cleandata_path + "combined.json")
tie_test = TemporalDataset(cleandata_path + "test.json")
tie_sampler = EpochRandomSubsetSampler(tie_train, EEPOCH_SIZE)
tie_train_loader = DataLoader(tie_train, batch_size=BATCH_SIZE, collate_fn=tie_collator, sampler=tie_sampler)
tie_test_loader = DataLoader(tie_test, batch_size=BATCH_SIZE, shuffle=False, collate_fn=tie_collator)

cleandata_path = "D:\\GeoTKG\\cleandata\\geo\\"
geo_train = GeoDataset(cleandata_path + "train.json")
geo_eval = GeoDataset(cleandata_path + "eval.json")
geo_sampler = EpochRandomSubsetSampler(geo_train, EEPOCH_SIZE)
geo_train_loader = DataLoader(geo_train, batch_size=BATCH_SIZE, collate_fn=geo_collator, sampler=geo_sampler)
geo_eval_loader = DataLoader(geo_eval, batch_size=BATCH_SIZE, shuffle=False, collate_fn=geo_collator)

cleandata_path = "D:\\GeoTKG\\cleandata\\normalise\\"
norm_train = TemporalDataset(cleandata_path + "combined.json")
norm_test = TemporalDataset(cleandata_path + "test.json")
norm_sampler = EpochRandomSubsetSampler(norm_train, EEPOCH_SIZE)
norm_train_loader = DataLoader(norm_train, batch_size=BATCH_SIZE, collate_fn=norm_collator, sampler=tie_sampler)
norm_test_loader = DataLoader(norm_test, batch_size=BATCH_SIZE, shuffle=False, collate_fn=norm_collator)

In [4]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

model = GeoTKG().to(device)

optimizer = AdamW([
        {"params": [p for n,p in model.named_parameters() if n.startswith("tie_model.enc.") or n.startswith("geo_model.enc.") or n.startswith("norm_model.")], "lr": ENC_LR},
        {"params": [p for n,p in model.named_parameters() if not n.startswith("tie_model.enc.") and not n.startswith("geo_model.enc.") and not n.startswith("norm_model.")], "lr": NONENC_LR},
    ], weight_decay=0.01)

steps_per_epoch = len(tie_train_loader)
num_train_steps = steps_per_epoch * NUM_EPOCHS
num_warmup_steps = steps_per_epoch * WARMUP_EPOCHS

sched = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_train_steps)

scaler = torch.amp.GradScaler(enabled=(device.type=='cuda'))

Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.weight', 'lm_head.layer_norm.bias', 'lm_head.dense.bias']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
d:\GeoTKG\venv\Lib\site-packages

In [5]:
global_step = 0
history = {#  Training
           "loss": [], "tie_loss":[], "norm_loss":[], "geo_loss":[],
           #  Evaluation
           "et_ner_f1": [], 'geo_ner_f1':[], "et_f1": [], "ee_f1": [], "norm_strict":[], "norm_relaxed":[], 
           "eval_loss":[], "eval_tie_loss":[], "eval_norm_loss":[], "eval_geo_loss":[]}

for epoch in range(NUM_EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss_sum = 0
    tie_loss_sum = 0
    norm_loss_sum = 0
    geo_loss_sum = 0
    for step, (tie_batch, geo_batch, norm_batch) in enumerate(zip(tie_train_loader, geo_train_loader, norm_train_loader)):
        tie_batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in tie_batch.items()}
        geo_batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in geo_batch.items()}
        norm_batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in norm_batch.items()}
        out = model(tie_batch, geo_batch, norm_batch)
        loss = out["loss"]
        loss_sum += loss.item()
        tie_loss_sum += out["tie_loss"].item()
        norm_loss_sum += out["norm_loss"].item()
        geo_loss_sum += out["geo_loss"].item()

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        prev_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        sched.step() if scaler.get_scale() <= prev_scale else None
        optimizer.zero_grad(set_to_none=True)
        global_step += 1

    print(f"Epoch {epoch+1} done. Avg Loss: {loss_sum / steps_per_epoch:.4f}")

    # ---- validation ----
    if (epoch + 1) % LOGGING_EPOCHS == 0:
        model.eval()
        with torch.no_grad():
            batch_evaluation = model.evaluate_dataloader(tie_test_loader, geo_eval_loader, norm_test_loader)
            history["et_ner_f1"].append(batch_evaluation["et_ner_f1"])
            history["geo_ner_f1"].append(batch_evaluation["geo_ner_f1"])
            history["norm_strict"].append(batch_evaluation["norm_strict"])
            history["norm_relaxed"].append(batch_evaluation["norm_relaxed"])
            history["et_f1"].append(batch_evaluation["et_f1"])
            history["ee_f1"].append(batch_evaluation["ee_f1"])
            history["eval_loss"].append(batch_evaluation["eval_loss"])
            history["eval_tie_loss"].append(batch_evaluation["eval_tie_loss"])
            history["eval_geo_loss"].append(batch_evaluation["eval_geo_loss"])
            history["eval_norm_loss"].append(batch_evaluation["eval_norm_loss"])
            history["loss"].append(loss_sum / steps_per_epoch)
            history["tie_loss"].append(tie_loss_sum / steps_per_epoch)
            history["geo_loss"].append(geo_loss_sum / steps_per_epoch)
            history["norm_loss"].append(norm_loss_sum / steps_per_epoch)
        model.save(f"D:\\GeoTKG\\training\\results\\geotkg\\geotkg_model_epoch{epoch+1}.pt")
        print(f'''EPOCH{epoch+1} \n
              ET NER F1={batch_evaluation['et_ner_f1']:.4f}, GEO NER F1={batch_evaluation['geo_ner_f1']:.4f}, \n
              ET F1={batch_evaluation['et_f1']:.4f}, EE F1={batch_evaluation['ee_f1']:.4f}, \n
              Norm Strict={batch_evaluation['norm_strict']}, Norm Relax={batch_evaluation['norm_relaxed']}\n
              Train Loss={loss_sum / steps_per_epoch:.4f}, Eval Loss={batch_evaluation['eval_loss']:.4f}\n
              Tie Loss ({tie_loss_sum / steps_per_epoch:.4f}, {batch_evaluation["eval_tie_loss"]:.4f}), Geo Loss ({geo_loss_sum / steps_per_epoch:.4f}, {batch_evaluation["eval_geo_loss"]:.4f}), Norm Loss ({norm_loss_sum / steps_per_epoch:.4f}, {norm_loss_sum / steps_per_epoch})''')

KeyboardInterrupt: 

In [ ]:
import json
with open("results/geotkg_model/history.json", "w") as f:
    json.dump(history, f, indent=2)

In [ ]:
import matplotlib.pyplot as plt

x = list(range(5, 21, 5))

plt.plot(x, history["ner_f1"], label="NER F1")
plt.plot(x, history["et_f1"], label="ET F1")
plt.plot(x, history["ee_f1"], label="EE F1")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Evaluation Metrics Over Time")
plt.legend()
plt.show()

In [ ]:
plt.plot(x, history["loss"], label="Loss")
plt.plot(x, history["ner_loss"], label="NER Loss")
plt.plot(x, history["ee_loss"], label="EE Loss")
plt.plot(x, history["et_loss"], label="ET Loss")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.show()

In [ ]:
plt.plot(x, history["eval_loss"], label="Loss")
plt.plot(x, history["ner_eval_loss"], label="NER Loss")
plt.plot(x, history["ee_eval_loss"], label="EE Loss")
plt.plot(x, history["et_eval_loss"], label="ET Loss")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Evaluation Loss")
plt.show()

In [ ]:
plt.plot(x, history["eval_loss"], label="Eval Loss")
plt.plot(x, history["loss"], label="Train Loss")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Compare Loss")
plt.show()

In [ ]:
plt.plot(x, history["ee_eval_loss"], label="Eval Loss")
plt.plot(x, history["ee_loss"], label="Train Loss")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Compare EE Loss")
plt.show()

In [ ]:
TIEModel = TIEModel()
load = torch.load("D:\\GeoTKG\\geotkg\\results\\tie_model\\tie_model_epoch20.pt")
TIEModel.load_state_dict(state_dict=load['model_state_dict'])

In [ ]:
metrics, et_ner_pred_ids = TIEModel.evaluate_dataloader(eval_loader, return_ner_tags=True)

In [ ]:
len(et_ner_pred_ids)

In [ ]:
et_ner_pred_ids.tolist()

In [ ]:
import json
with open("geo-tkg-et-ner-preds.json", 'w') as json_file:
    for sample in et_ner_pred_ids:
        sample=sample.tolist()
        json_file.write(json.dumps(sample)+"\n")